# Notebook 03 — Slab Construction: Hastelloy N (111) Surface
## Hastelloy N | Hydrogen Interaction Study


**This notebook covers:**
- Theory behind slab models and why (111) is the correct surface for FCC Hastelloy N
- FCC (111) surface geometry, stacking sequence, and layer spacing
- Slab thickness and vacuum requirements
- Building the (111) slab from the minimized bulk using ASE
- Size and thickness analysis
- Surface layer composition identification
- Writing output files with complete Masses section


---
## 2. Execution Strategy
---

### What We Are Doing in This Notebook

**Goal:** Build a physically correct 3×3×6 Hastelloy N (111) slab with 15 Å vacuum, verify its geometry, identify the surface layer composition, and write output files ready for LAMMPS.

**Steps:**
```
3.1  Imports + parameters  ← a₀ = 3.5747 Å from Notebook 02 NPT
3.2  Size & thickness analysis of input bulk
3.3  Build (111) slab with ASE fcc111()
3.4  Substitute atoms to match Hastelloy N composition
3.5  Set initial magnetic moments
3.6  Slab thickness analysis — layers, spacing, vacuum, cell height
3.7  Identify surface layer atoms and composition
3.8  Write output files: .xyz + .lammps (with complete Masses section)
3.9  Structure summary
```

---
## 3. Code
---

### 3.1 — Imports & Parameters

In [6]:
import numpy as np
from ase.build import fcc111
from ase.io import read, write
from ase.data import atomic_masses, chemical_symbols
from collections import Counter
import os

import ase
print(f'ASE version : {ase.__version__}')
print(f'NumPy       : {np.__version__}')

# ═══════════════════════════════════════════════════════════
# PARAMETERS — edit here if anything changes
# ═══════════════════════════════════════════════════════════

# Lattice parameter — NPT-confirmed value at 300 K from Notebook 02
A0           = 3.5208   # Å — MACE-MP-0b2 equilibrium at 300 K [Notebook 02]

# Slab geometry
SURFACE      = (1, 1, 1)  # Miller indices — (111) FCC surface
NX, NY       = 5, 6       # lateral supercell size (5×6 = 30 atoms per layer)
N_LAYERS     = 12          # atomic layers —  for bulk-like interior [1,2,12]
VACUUM       = 15.0       # Å — vacuum gap above surface [1,2,3]

# Reproducibility — same seed as Notebook 01 [22]
RANDOM_SEED  = 7

# Hastelloy N composition
# 5×6×12 slab = 360 atoms total
COMPOSITION = {
    'Ni': 260,   # 72% — austenitic FCC matrix
    'Mo': 36,   # 10% — solid-solution strengthener, corrosion resistance
    'Cr': 30,   #  8.4% — oxidation / corrosion resistance
    'Fe': 13,   #  3.6% — residual from raw materials
    'Al': 7,   #  2% — deoxidiser, minor γ' former
    'C':  7,   #  2% — carbide former (M₆C, M₂₃C₆) [1,4]
    'B':  7,   #  2% — grain boundary strengthener, boride former [1,4]
}
# Atom type mapping — alphabetical, must match pair_coeff [15,16]
# Type 1=Al, 2=B, 3=C, 4=Cr, 5=Fe, 6=Mo, 7=Ni, 8=H (future)
MASSES = {
    1: (26.9815, 'Al'),
    2: (10.8110, 'B'),
    3: (12.0110, 'C'),
    4: (51.9961, 'Cr'),
    5: (55.8450, 'Fe'),
    6: (95.9600, 'Mo'),
    7: (58.6934, 'Ni'),
    8: ( 1.0080, 'H'),   # no H atoms yet
}

# Magnetic moments — same values as Notebook 01 [Ref NB01]
MAGMOM_MAP = {
    'Ni': 0.71, 'Mo': 0.0, 'Cr': 0.0, 'Fe': 2.64,
    'Al': 0.0, 'C': 0.0,  'B': 0.0,  'H': 0.0,
}

# Paths
BULK_INPUT  = 'structures/02_bulk_energy_minimization/Hastelloy_N_7_minimized.lammps'
SLAB_XYZ    = 'structures/notebook03-Slab-generation/7/hastelloy_7_slab.xyz'
SLAB_LAMMPS = 'structures/notebook03-Slab-generation/7/hastelloy_7_slab.lammps'

os.makedirs('structures/notebook03-Slab-generation/7', exist_ok=True)

# Verify input exists
if not os.path.exists(BULK_INPUT):
    raise FileNotFoundError(f'{BULK_INPUT} not found. Complete Bulk minimization first.')

# Verify composition
total = sum(COMPOSITION.values())
expected = NX * NY * N_LAYERS
if total != expected:
    raise ValueError(f'Composition sums to {total}, expected {expected} for {NX}×{NY}×{N_LAYERS} slab.')

print(f'\nSurface      : ({SURFACE[0]}{SURFACE[1]}{SURFACE[2]}) FCC  [4,5]')
print(f'Lateral size : {NX}×{NY}')
print(f'Layers       : {N_LAYERS}  [1,2,12]')
print(f'Vacuum       : {VACUUM} Å  [1,2,3]')
print(f'Total atoms  : {total}')
print(f'a₀ (300 K)   : {A0} Å  [Notebook 02 NPT]')
print(f'Random seed  : {RANDOM_SEED}  [22]')

ASE version : 3.27.0
NumPy       : 2.4.3

Surface      : (111) FCC  [4,5]
Lateral size : 5×6
Layers       : 12  [1,2,12]
Vacuum       : 15.0 Å  [1,2,3]
Total atoms  : 360
a₀ (300 K)   : 3.5208 Å  [Notebook 02 NPT]
Random seed  : 7  [22]


### 3.2 — Size & Thickness Analysis: Input Bulk

In [7]:
# ─────────────────────────────────────────────────────────────
# SIZE & THICKNESS ANALYSIS — Input bulk (from Notebook 02)
# ─────────────────────────────────────────────────────────────

bulk = read(BULK_INPUT, format='lammps-data')
L_bulk = bulk.cell.lengths()

# FCC (111) layer spacing: d_111 = a0 / sqrt(3)  [4,11]
d111 = A0 / np.sqrt(3)

# Expected slab dimensions
# (111) surface lateral vectors: |a1| = |a2| = a0/sqrt(2)
a_surf = A0 / np.sqrt(2)   # surface lattice parameter
slab_x = NX * a_surf       # lateral x dimension
slab_y = NY * a_surf       # lateral y dimension  (not exactly — hex geometry)
slab_z_metal = (N_LAYERS - 1) * d111   # slab thickness (metal only)
slab_z_total = slab_z_metal + VACUUM   # total cell height

print('=' * 62)
print('  SIZE & THICKNESS ANALYSIS — Notebook 03')
print('=' * 62)
print(f'  Input bulk (Notebook 02):')
print(f'    Atoms              : {len(bulk)}')
print(f'    Box a (Å)          : {L_bulk[0]:.4f}')
print(f'    Box b (Å)          : {L_bulk[1]:.4f}')
print(f'    Box c (Å)          : {L_bulk[2]:.4f}')
print()
print(f'  FCC (111) geometry [4,11]:')
print(f'    a₀ (300 K NPT)     : {A0:.4f} Å')
print(f'    d₁₁₁ = a₀/√3      : {d111:.4f} Å  (interlayer spacing)')
print(f'    a_surface = a₀/√2  : {a_surf:.4f} Å  (surface lattice parameter)')
print()
print(f'  Planned slab ({NX}×{NY}×{N_LAYERS}):')
print(f'    Atoms per layer    : {NX * NY}')
print(f'    Total atoms        : {NX * NY * N_LAYERS}')
print(f'    Slab thickness     : {slab_z_metal:.3f} Å  ({N_LAYERS-1} × d₁₁₁)')
print(f'    Vacuum gap         : {VACUUM:.1f} Å  [1,2,3]')
print(f'    Total cell z       : {slab_z_total:.3f} Å')
print()
print(f'  Requirement checks:')
# Vacuum > 2 × MACE cutoff
MACE_RCUT = 6.0
vac_ok = VACUUM > 2 * MACE_RCUT
print(f'    Vacuum > 2×rcut ({2*MACE_RCUT:.0f} Å): {VACUUM:.0f} Å  {"✓" if vac_ok else "✗"}')
# Slab > minimum 4 layers
slab_ok = N_LAYERS >= 6
print(f'    Layers ≥ 6 (min adequate [12]): {N_LAYERS}  {"✓" if slab_ok else "✗"}')
# Lateral size > 2 × rcut (to avoid H-H periodic image interactions)
lat_ok = slab_x > 2 * MACE_RCUT
print(f'    Lateral > 2×rcut ({2*MACE_RCUT:.0f} Å): {slab_x:.2f} Å  {"✓" if lat_ok else "✗"}')
print()
if all([vac_ok, slab_ok, lat_ok]):
    print('  ✓ All size requirements satisfied')
else:
    print('  ✗ Size requirement(s) failed — adjust parameters')
print('=' * 62)

  SIZE & THICKNESS ANALYSIS — Notebook 03
  Input bulk (Notebook 02):
    Atoms              : 500
    Box a (Å)          : 17.8618
    Box b (Å)          : 17.8618
    Box c (Å)          : 17.8618

  FCC (111) geometry [4,11]:
    a₀ (300 K NPT)     : 3.5208 Å
    d₁₁₁ = a₀/√3      : 2.0327 Å  (interlayer spacing)
    a_surface = a₀/√2  : 2.4896 Å  (surface lattice parameter)

  Planned slab (5×6×12):
    Atoms per layer    : 30
    Total atoms        : 360
    Slab thickness     : 22.360 Å  (11 × d₁₁₁)
    Vacuum gap         : 15.0 Å  [1,2,3]
    Total cell z       : 37.360 Å

  Requirement checks:
    Vacuum > 2×rcut (12 Å): 15 Å  ✓
    Layers ≥ 6 (min adequate [12]): 12  ✓
    Lateral > 2×rcut (12 Å): 12.45 Å  ✓

  ✓ All size requirements satisfied


### 3.3 — Build (111) Slab with ASE

> `fcc111()` builds the correct FCC (111) geometry with ABCABC stacking and hexagonal surface unit cell [14]. We use it with our MACE-confirmed a₀ = 3.5747 Å.

In [8]:
# ─────────────────────────────────────────────────────────────
# Build FCC (111) slab using ASE fcc111() [14]
#
# fcc111 parameters:
#   symbol  : base element for geometry (we use 'Ni' — substituted below)
#   size    : (nx, ny, nlayers) — lateral supercell × layer count
#   a       : lattice parameter (Å) — use NPT-confirmed 300 K value
#   vacuum  : vacuum thickness (Å) — added ABOVE the top surface
#   orthogonal: True → forces orthogonal cell (needed for LAMMPS)
#   periodic  : False → non-periodic in z (slab geometry)
# ─────────────────────────────────────────────────────────────

slab = fcc111(
    symbol     = 'Ni',
    size       = (NX, NY, N_LAYERS),
    a          = A0,
    vacuum     = VACUUM,
    orthogonal = True,
    periodic   = True,
)

# Verify structure
L = slab.cell.lengths()
pos = slab.get_positions()
print(f'Slab built successfully:')
print(f'  Total atoms      : {len(slab)}')
print(f'  Expected         : {NX * NY * N_LAYERS}')
print(f'  Cell a (Å)       : {L[0]:.4f}')
print(f'  Cell b (Å)       : {L[1]:.4f}')
print(f'  Cell c (Å)       : {L[2]:.4f}  (slab + vacuum)')
print(f'  PBC              : {slab.get_pbc()}')
print()

# Check all atoms are Ni at this stage
assert all(s == 'Ni' for s in slab.get_chemical_symbols()), \
    'Expected all Ni before substitution'
print(f'  All {len(slab)} atoms are Ni — substitution in next cell ✓')
print(f'  Boundary: periodic in x,y — non-periodic in z (slab model) ✓')

Slab built successfully:
  Total atoms      : 360
  Expected         : 360
  Cell a (Å)       : 12.4479
  Cell b (Å)       : 12.9362
  Cell c (Å)       : 52.3601  (slab + vacuum)
  PBC              : [ True  True  True]

  All 360 atoms are Ni — substitution in next cell ✓
  Boundary: periodic in x,y — non-periodic in z (slab model) ✓


In [9]:
from ase.visualize import view
view(slab, viewer='ngl')

### 3.4 — Random Substitution for Hastelloy N Composition

In [10]:
# ─────────────────────────────────────────────────────────────
# Random substitution — same method as Notebook 01 [9,22]
# Same seed (42) for statistical consistency
# ─────────────────────────────────────────────────────────────

np.random.seed(RANDOM_SEED)
print(f'Random seed: {RANDOM_SEED}  (same as Notebook 01 — document in lab notes) [22]')

all_indices = np.arange(len(slab))
np.random.shuffle(all_indices)

symbols = ['Ni'] * len(slab)
pointer = 0
substitution_log = []
for element, count in COMPOSITION.items():
    if element == 'Ni':
        continue
    idx = all_indices[pointer:pointer + count]
    for i in idx:
        symbols[i] = element
    substitution_log.append((element, count, idx.tolist()))
    pointer += count

slab.set_chemical_symbols(symbols)

actual = Counter(slab.get_chemical_symbols())
print(f'\n{"Element":>8s} {"Actual":>8s} {"Target":>8s} {"OK?":>5s}')
print('-' * 35)
all_ok = True
for elem in COMPOSITION:
    a, t = actual.get(elem, 0), COMPOSITION[elem]
    ok = '✓' if a == t else '✗'
    if a != t: all_ok = False
    print(f'{elem:>8s} {a:8d} {t:8d} {ok:>5s}')
print()
print('✓ Composition correct' if all_ok else '✗ MISMATCH — check COMPOSITION dict')

Random seed: 7  (same as Notebook 01 — document in lab notes) [22]

 Element   Actual   Target   OK?
-----------------------------------
      Ni      260      260     ✓
      Mo       36       36     ✓
      Cr       30       30     ✓
      Fe       13       13     ✓
      Al        7        7     ✓
       C        7        7     ✓
       B        7        7     ✓

✓ Composition correct


### 3.5 — Set Initial Magnetic Moments

In [11]:
# ─────────────────────────────────────────────────────────────
# Set initial magnetic moments — same values as Notebook 01
# Required by MACE-MP-0b2 [13,15]
# Ni: 0.606 μ_B [Ref NB01-31]; Fe: 2.22 μ_B [Ref NB01-34]
# ─────────────────────────────────────────────────────────────

magmoms = [MAGMOM_MAP.get(s, 0.0) for s in slab.get_chemical_symbols()]
slab.set_initial_magnetic_moments(magmoms)

print('Magnetic moments set:')
for elem, mu in MAGMOM_MAP.items():
    n = actual.get(elem, 0)
    if n > 0:
        print(f'  {elem:2s}: {mu:.1f} μ_B × {n} atoms = {mu*n:.1f} μ_B')
print(f'\n✓ Magnetic moments assigned')

Magnetic moments set:
  Ni: 0.7 μ_B × 260 atoms = 184.6 μ_B
  Mo: 0.0 μ_B × 36 atoms = 0.0 μ_B
  Cr: 0.0 μ_B × 30 atoms = 0.0 μ_B
  Fe: 2.6 μ_B × 13 atoms = 34.3 μ_B
  Al: 0.0 μ_B × 7 atoms = 0.0 μ_B
  C : 0.0 μ_B × 7 atoms = 0.0 μ_B
  B : 0.0 μ_B × 7 atoms = 0.0 μ_B

✓ Magnetic moments assigned


### 3.6 — Slab Thickness Analysis

> This is the critical verification step for every slab calculation. We confirm the number of layers, layer spacing, slab thickness, vacuum thickness, and total cell height all match expectations.

In [12]:
# ─────────────────────────────────────────────────────────────
# SLAB THICKNESS ANALYSIS
# Refs: slab requirements [1,2,12]; vacuum requirement [1,2,3]
# ─────────────────────────────────────────────────────────────

pos      = slab.get_positions()
z_coords = pos[:, 2]
L        = slab.cell.lengths()

# Identify distinct atomic layers by clustering z-coordinates
# Round to 2 decimal places to group atoms in the same layer
z_rounded = np.round(z_coords, decimals=2)
layer_z   = np.sort(np.unique(z_rounded))
n_layers_found = len(layer_z)

# Layer spacings
layer_spacings = np.diff(layer_z)
mean_spacing   = np.mean(layer_spacings)
expected_d111  = A0 / np.sqrt(3)

# Slab thickness = from bottom atomic layer to top atomic layer
z_metal_min  = z_coords.min()
z_metal_max  = z_coords.max()
slab_thickness = z_metal_max - z_metal_min

# Vacuum thickness = cell_z - top_atom_z (vacuum above top surface)
cell_z        = L[2]
vacuum_actual = cell_z - z_metal_max

# Atoms per layer
atoms_per_layer = []
for z in layer_z:
    count = np.sum(np.abs(z_rounded - z) < 0.01)
    atoms_per_layer.append(count)

print('=' * 65)
print('  SLAB THICKNESS ANALYSIS')
print(f'  Surface: ({SURFACE[0]}{SURFACE[1]}{SURFACE[2]}) FCC  |  Size: {NX}×{NY}×{N_LAYERS}  |  a₀ = {A0} Å')
print('=' * 65)
print(f'  Atoms total            : {len(slab)}')
print(f'  Layers found           : {n_layers_found}  (expected: {N_LAYERS})')
print(f'  Atoms per layer        : {atoms_per_layer}  (expected: {NX*NY} each)')
print()
print(f'  Layer z-positions (Å):')
for i, (z, n) in enumerate(zip(layer_z, atoms_per_layer), 1):
    role = '← TOP SURFACE (active)' if i == n_layers_found else \
           '← bottom (fixed in NB04)' if i == 1 else \
           '← bottom-1 (fixed in NB04)' if i == 2 else ''
    print(f'    Layer {i}: z = {z:.4f} Å  ({n} atoms)  {role}')
print()
print(f'  Interlayer spacings (Å): {[f"{s:.4f}" for s in layer_spacings]}')
print(f'  Mean spacing           : {mean_spacing:.4f} Å')
print(f'  Expected d₁₁₁=a₀/√3   : {expected_d111:.4f} Å  [4,11]')
print(f'  Deviation              : {abs(mean_spacing-expected_d111):.4f} Å')
print()
print(f'  Cell c (total)         : {cell_z:.4f} Å')
print(f'  Slab thickness         : {slab_thickness:.4f} Å  ({n_layers_found-1} × d₁₁₁)')
print(f'  Vacuum (above surface) : {vacuum_actual:.4f} Å  (target: {VACUUM} Å)')
print()

# Checks
checks = [
    (f'Correct layer count ({N_LAYERS})', n_layers_found == N_LAYERS),
    (f'Layer spacing ≈ d₁₁₁ ({expected_d111:.3f} Å)', abs(mean_spacing - expected_d111) < 0.05),
    (f'Vacuum ≥ 15 Å ({vacuum_actual:.2f} Å)', vacuum_actual >= 14.5),
    (f'All layers have {NX*NY} atoms', all(n == NX*NY for n in atoms_per_layer)),
]
for label, ok in checks:
    print(f'  {"✓" if ok else "✗"} {label}')
print('=' * 65)

  SLAB THICKNESS ANALYSIS
  Surface: (111) FCC  |  Size: 5×6×12  |  a₀ = 3.5208 Å
  Atoms total            : 360
  Layers found           : 12  (expected: 12)
  Atoms per layer        : [np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30), np.int64(30)]  (expected: 30 each)

  Layer z-positions (Å):
    Layer 1: z = 15.0000 Å  (30 atoms)  ← bottom (fixed in NB04)
    Layer 2: z = 17.0300 Å  (30 atoms)  ← bottom-1 (fixed in NB04)
    Layer 3: z = 19.0700 Å  (30 atoms)  
    Layer 4: z = 21.1000 Å  (30 atoms)  
    Layer 5: z = 23.1300 Å  (30 atoms)  
    Layer 6: z = 25.1600 Å  (30 atoms)  
    Layer 7: z = 27.2000 Å  (30 atoms)  
    Layer 8: z = 29.2300 Å  (30 atoms)  
    Layer 9: z = 31.2600 Å  (30 atoms)  
    Layer 10: z = 33.2900 Å  (30 atoms)  
    Layer 11: z = 35.3300 Å  (30 atoms)  
    Layer 12: z = 37.3600 Å  (30 atoms)  ← TOP SURFACE (active)

  Interlayer spacings (Å): ['2

### 3.7 — Identify Surface Layer Composition

> The top surface layer composition matters for adsorption studies — different elements have different binding strengths for H₂. This cell identifies exactly which atoms are at the surface.

In [8]:
# ─────────────────────────────────────────────────────────────
# Identify surface layer atoms
# Surface = topmost atomic layer (highest z-coordinate)
# Subsurface = second layer from top
# ─────────────────────────────────────────────────────────────

syms  = np.array(slab.get_chemical_symbols())
z_all = slab.get_positions()[:, 2]

# Top surface layer (layer N_LAYERS)
z_top      = layer_z[-1]
surf_mask  = np.abs(z_all - z_top) < 0.1
surf_syms  = syms[surf_mask]
surf_comp  = Counter(surf_syms)
surf_idx   = np.where(surf_mask)[0]

# Second layer from top (subsurface)
z_sub      = layer_z[-2]
sub_mask   = np.abs(z_all - z_sub) < 0.1
sub_syms   = syms[sub_mask]
sub_comp   = Counter(sub_syms)

# Bottom layer (will be fixed in Notebook 04)
z_bot      = layer_z[0]
bot_mask   = np.abs(z_all - z_bot) < 0.1
bot_syms   = syms[bot_mask]
bot_comp   = Counter(bot_syms)

print('Surface Layer Composition Analysis')
print('=' * 55)
print(f'\nTOP SURFACE (z = {z_top:.3f} Å) — {sum(surf_comp.values())} atoms:')
print(f'  These atoms are directly exposed to vacuum')
print(f'  H₂ will adsorb here in Notebook 05')
for elem in ['Ni','Mo','Cr','Fe','Al','C','B']:
    n = surf_comp.get(elem, 0)
    if n > 0:
        pct = n / sum(surf_comp.values()) * 100
        print(f'    {elem:2s}: {n} atoms  ({pct:.1f}%)')

print(f'\nSUBSURFACE (z = {z_sub:.3f} Å) — {sum(sub_comp.values())} atoms:')
print(f'  H atoms will occupy interstitial sites here (Notebooks 08-09)')
for elem in ['Ni','Mo','Cr','Fe','Al','C','B']:
    n = sub_comp.get(elem, 0)
    if n > 0:
        pct = n / sum(sub_comp.values()) * 100
        print(f'    {elem:2s}: {n} atoms  ({pct:.1f}%)')

print(f'\nBOTTOM LAYER (z = {z_bot:.3f} Å) — {sum(bot_comp.values())} atoms:')
print(f'  Will be FIXED in Notebook 04 to mimic bulk constraint')
for elem in ['Ni','Mo','Cr','Fe','Al','C','B']:
    n = bot_comp.get(elem, 0)
    if n > 0:
        print(f'    {elem:2s}: {n} atoms')

print(f'\nSurface atom indices (0-indexed, for reference in Notebook 05):')
print(f'  {surf_idx.tolist()}')
print('=' * 55)

Surface Layer Composition Analysis

TOP SURFACE (z = 37.360 Å) — 30 atoms:
  These atoms are directly exposed to vacuum
  H₂ will adsorb here in Notebook 05
    Ni: 21 atoms  (70.0%)
    Mo: 4 atoms  (13.3%)
    Cr: 2 atoms  (6.7%)
    Fe: 1 atoms  (3.3%)
    C : 1 atoms  (3.3%)
    B : 1 atoms  (3.3%)

SUBSURFACE (z = 35.330 Å) — 30 atoms:
  H atoms will occupy interstitial sites here (Notebooks 08-09)
    Ni: 20 atoms  (66.7%)
    Mo: 3 atoms  (10.0%)
    Cr: 5 atoms  (16.7%)
    C : 1 atoms  (3.3%)
    B : 1 atoms  (3.3%)

BOTTOM LAYER (z = 15.000 Å) — 30 atoms:
  Will be FIXED in Notebook 04 to mimic bulk constraint
    Ni: 23 atoms
    Mo: 3 atoms
    Cr: 1 atoms
    Fe: 1 atoms
    B : 2 atoms

Surface atom indices (0-indexed, for reference in Notebook 05):
  [330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359]


### 3.8 — Write Output Files

> Same direct file writing approach as Notebook 01 — all 8 masses including H written explicitly. The LAMMPS data file uses `boundary p p f` (non-periodic in z) for the slab.

In [9]:
# ─────────────────────────────────────────────────────────────
# Write output files
# Same direct writer as Notebook 01 — guarantees complete Masses
# ─────────────────────────────────────────────────────────────

syms_list = slab.get_chemical_symbols()
pos_arr   = slab.get_positions()
L         = slab.cell.lengths()

# Element to type number (alphabetical — must match pair_coeff)
present_elems = sorted(set(syms_list))
elem_to_type  = {el: i+1 for i, el in enumerate(present_elems)}

# ── XYZ for OVITO [17] ────────────────────────────────────────
write(SLAB_XYZ, slab)
print(f'Written : {SLAB_XYZ}  → open in OVITO [17]')

# ── LAMMPS data file ──────────────────────────────────────────
# Note: boundary p p f (non-periodic in z) is set in the LAMMPS
# INPUT SCRIPT, not in the data file itself. The data file just
# provides atom positions and box dimensions.
with open(SLAB_LAMMPS, 'w') as f:
    # Header
    f.write('# Hastelloy N (111) slab — written by Notebook 03\n')
    f.write(f'# a0={A0} Ang (300K NPT), {NX}x{NY}x{N_LAYERS} slab, {VACUUM}A vacuum\n\n')
    f.write(f'{len(slab)} atoms\n')
    f.write(f'{len(MASSES)} atom types\n\n')
    # Box — note z includes vacuum
    f.write(f'0.0  {L[0]:.10f}  xlo xhi\n')
    f.write(f'0.0  {L[1]:.10f}  ylo yhi\n')
    f.write(f'0.0  {L[2]:.10f}  zlo zhi\n\n')
    # Masses — all 8 types before Atoms section [15,16]
    f.write('Masses\n\n')
    for t, (m, el) in MASSES.items():
        f.write(f'{t}  {m}  # {el}\n')
    # Atoms
    f.write('\nAtoms # atomic\n\n')
    for i, (sym, p) in enumerate(zip(syms_list, pos_arr), 1):
        t = elem_to_type[sym]
        f.write(f'{i}  {t}  {p[0]:.10f}  {p[1]:.10f}  {p[2]:.10f}\n')

print(f'Written : {SLAB_LAMMPS}  → LAMMPS: read_data {SLAB_LAMMPS}')

# ── Verification ──────────────────────────────────────────────
print(f'\nAtom type mapping (must match pair_coeff in Notebooks 04+):')
for t, (m, el) in MASSES.items():
    n = syms_list.count(el) if el in syms_list else 0
    note = '← no atoms yet' if n == 0 else ''
    print(f'  Type {t}: {el:2s}  mass={m:.4f}  count={n}  {note}')

# ── Provenance record ─────────────────────────────────────────
record_path = 'structures/notebook03-Slab-generation/7/slab_generation_7_record.txt'
with open(record_path, 'w') as f:
    f.write('Hastelloy N (111) Slab — Generation Record\n')
    f.write('=' * 48 + '\n')
    f.write(f'Notebook       : 03_slab_construction.ipynb\n')
    f.write(f'Random seed    : {RANDOM_SEED}\n')
    f.write(f'a0 (300K NPT)  : {A0} Angstrom\n')
    f.write(f'Surface        : (111) FCC\n')
    f.write(f'Size           : {NX}x{NY}x{N_LAYERS}\n')
    f.write(f'Vacuum         : {VACUUM} Angstrom\n')
    f.write(f'Total atoms    : {len(slab)}\n')
    f.write(f'Box (Ang)      : {L[0]:.6f} x {L[1]:.6f} x {L[2]:.6f}\n')
    f.write(f'Atom types     : {len(MASSES)} (Al B C Cr Fe Mo Ni H)\n')
    f.write('\nComposition:\n')
    for elem, n in actual.items():
        f.write(f'  {elem}: {n} ({n/len(slab)*100:.1f}%)\n')
    f.write('\nSurface layer composition:\n')
    for elem, n in surf_comp.items():
        f.write(f'  {elem}: {n}\n')
    f.write('\nSubstituted indices (0-indexed):\n')
    for elem, count, idx in substitution_log:
        f.write(f'  {elem} ({count}): {idx}\n')
print(f'Written : {record_path}')
print('\n✓ All files written successfully')

Written : structures/notebook03-Slab-generation/7/hastelloy_7_slab.xyz  → open in OVITO [17]
Written : structures/notebook03-Slab-generation/7/hastelloy_7_slab.lammps  → LAMMPS: read_data structures/notebook03-Slab-generation/7/hastelloy_7_slab.lammps

Atom type mapping (must match pair_coeff in Notebooks 04+):
  Type 1: Al  mass=26.9815  count=7  
  Type 2: B   mass=10.8110  count=7  
  Type 3: C   mass=12.0110  count=7  
  Type 4: Cr  mass=51.9961  count=30  
  Type 5: Fe  mass=55.8450  count=13  
  Type 6: Mo  mass=95.9600  count=36  
  Type 7: Ni  mass=58.6934  count=260  
  Type 8: H   mass=1.0080  count=0  ← no atoms yet
Written : structures/notebook03-Slab-generation/7/slab_generation_7_record.txt

✓ All files written successfully


/home/akinyemi.az/miniforge3/envs/mace-lammps/lib/python3.11/site-packages/ase/io/extxyz.py:318: UserWarning: Skipping unhashable information adsorbate_info
  warnings.warn('Skipping unhashable information '


### 3.9 — Structure Summary

In [10]:
# ─────────────────────────────────────────────────────────────
# Final structure summary
# ─────────────────────────────────────────────────────────────

L   = slab.cell.lengths()
A   = slab.cell.angles()
S   = Counter(slab.get_chemical_symbols())

print('=' * 58)
print('  SLAB STRUCTURE SUMMARY')
print('=' * 58)
print(f'  Surface           : ({SURFACE[0]}{SURFACE[1]}{SURFACE[2]}) FCC — most stable [4,5]')
print(f'  Size              : {NX}×{NY}×{N_LAYERS}  ({NX*NY} atoms/layer × {N_LAYERS} layers)')
print(f'  Total atoms       : {len(slab)}')
print(f'  Cell a (Å)        : {L[0]:.4f}')
print(f'  Cell b (Å)        : {L[1]:.4f}')
print(f'  Cell c (Å)        : {L[2]:.4f}  (slab + vacuum)')
print(f'  Angles (°)        : α={A[0]:.1f} β={A[1]:.1f} γ={A[2]:.1f}')
print(f'  PBC               : {slab.get_pbc()}  (non-periodic in z)')
print(f'  Slab thickness    : {slab_thickness:.3f} Å')
print(f'  Vacuum            : {vacuum_actual:.3f} Å')
print(f'  d₁₁₁ (measured)   : {mean_spacing:.4f} Å  (expected: {expected_d111:.4f} Å) [4,11]')
print()
print('  Composition:')
for elem in ['Ni','Mo','Cr','Fe','Al','C','B']:
    n = S.get(elem, 0)
    if n > 0:
        print(f'    {elem:4s}: {n:3d} atoms  ({n/len(slab)*100:.1f}%)')
print()
print('  Layer structure (bottom → top):')
for i, (z, n) in enumerate(zip(layer_z, atoms_per_layer), 1):
    role = ' [SURFACE — active]' if i == len(layer_z) else \
           ' [fix in NB04]' if i <= 4 else ''
    print(f'    Layer {i}: z={z:.3f} Å, {n} atoms{role}')
print('=' * 58)

  SLAB STRUCTURE SUMMARY
  Surface           : (111) FCC — most stable [4,5]
  Size              : 5×6×12  (30 atoms/layer × 12 layers)
  Total atoms       : 360
  Cell a (Å)        : 12.4479
  Cell b (Å)        : 12.9362
  Cell c (Å)        : 52.3601  (slab + vacuum)
  Angles (°)        : α=90.0 β=90.0 γ=90.0
  PBC               : [ True  True  True]  (non-periodic in z)
  Slab thickness    : 22.360 Å
  Vacuum            : 15.000 Å
  d₁₁₁ (measured)   : 2.0327 Å  (expected: 2.0327 Å) [4,11]

  Composition:
    Ni  : 260 atoms  (72.2%)
    Mo  :  36 atoms  (10.0%)
    Cr  :  30 atoms  (8.3%)
    Fe  :  13 atoms  (3.6%)
    Al  :   7 atoms  (1.9%)
    C   :   7 atoms  (1.9%)
    B   :   7 atoms  (1.9%)

  Layer structure (bottom → top):
    Layer 1: z=15.000 Å, 30 atoms [fix in NB04]
    Layer 2: z=17.030 Å, 30 atoms [fix in NB04]
    Layer 3: z=19.070 Å, 30 atoms [fix in NB04]
    Layer 4: z=21.100 Å, 30 atoms [fix in NB04]
    Layer 5: z=23.130 Å, 30 atoms
    Layer 6: z=25.160 Å, 30 